# Null Handling and Format Normalization in Structured Extraction

When extracting structured data from free-form text, two problems frequently break production pipelines:

1. **Missing fields** — Claude fills in plausible-sounding but fabricated values instead of acknowledging absence
2. **Inconsistent formats** — the same concept appears as `"fifty thousand"`, `"$80K"`, or `"80000"` across documents

This notebook walks through a three-stage evolution of a schema that solves both problems.

| Stage | Problem solved | Key technique |
|-------|---------------|---------------|
| 1. Base Prompt | Gets data out | `tool_use` + `required` |
| 2. Null Handling | Stops hallucination on missing fields | `"type": ["string", "null"]` + explicit null rule |
| 3. Format Normalization | Standardises output format | Format rules in `description` fields |

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## Stage 1: Base Prompt

The simplest approach: define a tool with `required` for all fields and let Claude extract.

This works when all fields are present in the source text — but when a field is **absent**,
`required` creates pressure on Claude to fill in a plausible-sounding value rather than
admit the field does not exist.

> **Hallucination risk**: Claude may invent a contact email or fabricate a salary figure
> because the schema marks those fields as mandatory.

In [ ]:
// Stage 1: all fields required — hallucination risk when data is absent
const baseApplicantTool: Anthropic.Tool = {
  name: 'extract_applicant',
  description: 'Extracts applicant information from job application text.',
  input_schema: {
    type: 'object',
    properties: {
      name:             { type: 'string', description: 'Applicant full name' },
      email:            { type: 'string', description: 'Contact email address' },
      years_experience: { type: 'number', description: 'Years of professional experience' },
      expected_salary:  { type: 'number', description: 'Expected annual salary in USD' },
    },
    required: ['name', 'email', 'years_experience', 'expected_salary'],
  },
};

// This input mentions only name and experience — email and salary are absent
const incompleteText = 'My name is Alex Chen. I have 5 years of experience in backend development.';

const stage1Response = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  tools: [baseApplicantTool],
  tool_choice: { type: 'tool', name: 'extract_applicant' },
  messages: [{ role: 'user', content: incompleteText }],
});

const stage1Block = stage1Response.content.find(b => b.type === 'tool_use');
if (stage1Block?.type === 'tool_use') {
  console.log('Stage 1 output (notice fabricated email and salary):');
  console.log(JSON.stringify(stage1Block.input, null, 2));
}

## Stage 2: Null Handling Instruction

Two changes prevent hallucination on missing fields:

1. **Replace `"type": "string"` with `"type": ["string", "null"]`** — valid JSON Schema syntax
   that makes `null` a legal value. Remove the field from `required` so Claude is not forced to fill it.
2. **Add an explicit rule in the system prompt**: `"If any field cannot be found in the text, return null.
   Do not guess or fabricate values."`

`null` is meaningful output — it tells downstream code the field was absent in the source,
which is far safer than a fabricated value that passes type-checking but carries wrong data.

```
JSON Schema  →  type: ["string", "null"] makes null legal
System prompt  →  explicit null rule tells Claude when to use it
```

Both layers are required: the schema alone does not instruct Claude *when* to use null;
the prompt rule alone cannot make null type-safe without the schema allowing it.

In [ ]:
// Stage 2: nullable types — Claude returns null instead of fabricating
const nullableApplicantTool: Anthropic.Tool = {
  name: 'extract_applicant',
  description: 'Extracts applicant information from job application text.',
  input_schema: {
    type: 'object',
    properties: {
      name:             { type: ['string', 'null'], description: 'Applicant full name' },
      email:            { type: ['string', 'null'], description: 'Contact email address' },
      years_experience: { type: ['number', 'null'], description: 'Years of professional experience' },
      expected_salary:  { type: ['number', 'null'], description: 'Expected annual salary in USD' },
    },
  },
};

const SYSTEM_PROMPT_NULL =
  'If any field cannot be found in the text, return null for that field. ' +
  'Do not guess or fabricate values.';

// Same incomplete input as Stage 1
const stage2Response = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  system: SYSTEM_PROMPT_NULL,
  tools: [nullableApplicantTool],
  tool_choice: { type: 'tool', name: 'extract_applicant' },
  messages: [{ role: 'user', content: incompleteText }],
});

const stage2Block = stage2Response.content.find(b => b.type === 'tool_use');
if (stage2Block?.type === 'tool_use') {
  type Applicant = {
    name: string | null;
    email: string | null;
    years_experience: number | null;
    expected_salary: number | null;
  };
  const data = stage2Block.input as Applicant;

  console.log('Stage 2 output (null for absent fields — no fabrication):');
  console.log(JSON.stringify(data, null, 2));

  // Downstream: check for null before using
  console.log('\n--- Downstream null-check example ---');
  if (data.email === null) {
    console.log('email: not provided — skip email validation step');
  }
  if (data.expected_salary === null) {
    console.log('expected_salary: not provided — flag for recruiter follow-up');
  }
}

## Stage 2 Supplement A: `null` vs Placeholder Strings

When a field is absent, Claude under `required` pressure will fabricate plausible-looking values.
A common variant is **placeholder strings** — `"UNKNOWN"`, `"N/A"`, `"TBD"` — which are slightly
more honest than a fabricated value, but still dangerous:

| Approach | Passes type check | Detectable downstream | Risk |
|----------|------------------|-----------------------|------|
| `null` | ✅ (with union type) | `value === null` always works | None — honest absence |
| `"UNKNOWN"` / `""` | ✅ (passes as `string`) | Must inspect string content | Silent contamination |
| Fabricated value | ✅ | Cannot detect without domain knowledge | Enters production undetected |

**The critical difference**: placeholder strings *pass format validation* and enter downstream
systems undetected — the same way a hallucinated email address does.

`null` is honest absence. A downstream system can branch on it reliably with a simple `=== null` check.

> **Exam connection**: this is exactly why Missing Information failures cannot be fixed by retrying —
> each retry produces a different placeholder (`"UNKNOWN"` → `"AGT-001"` → `"TBD"`), all of which
> pass format checks but carry wrong data.

In [ ]:
// null is reliably detectable downstream; placeholder strings are not

const withPlaceholders = {
  name: 'Alex Chen',
  email: 'alex.chen@placeholder.com',  // fabricated — passes string type-check
  years_experience: 5,
  expected_salary: 0,                  // fabricated — passes number type-check
};

const withNull = {
  name: 'Alex Chen',
  email: null,             // honest absence — detectable downstream
  years_experience: 5,
  expected_salary: null,   // honest absence
};

console.log('--- Placeholder strings (Stage 1 hallucination) ---');
console.log('email:', withPlaceholders.email);
console.log('email === null:', withPlaceholders.email === null, '← undetectable fabrication');
console.log('typeof email:', typeof withPlaceholders.email, '← passes string type-check');

console.log('\n--- null (Stage 2 output) ---');
console.log('email:', withNull.email);
console.log('email === null:', withNull.email === null, '← reliably detectable ✓');

if (withNull.email === null) {
  console.log('\nDownstream: email absent → flag for manual follow-up');
}
if (withNull.expected_salary === null) {
  console.log('Downstream: salary absent → recruiter to confirm');
}

## Stage 2 Supplement B: `"type": "null"` vs `"type": ["string", "null"]`

A frequent schema mistake: using the bare `"null"` type instead of the union form.

```json
// WRONG — field can ONLY be null; any real value Claude returns fails
{ "email": { "type": "null" } }

// CORRECT — both a string value and null are valid
{ "email": { "type": ["string", "null"] } }
```

The same union pattern applies to every base type:

| Nullable field | Correct syntax |
|----------------|----------------|
| text / identifier | `["string", "null"]` |
| decimal amount | `["number", "null"]` |
| integer count | `["integer", "null"]` |
| flag | `["boolean", "null"]` |

**Exam trap**: `"type": "null"` is syntactically valid JSON Schema — it just means the
field can *only* hold `null`, which is almost never the intended design.

In [ ]:
// Union type syntax: ["string", "null"] accepts string OR null
// Bare "null" type accepts ONLY null — a real string value would be invalid

const wrongSyntax = {
  // WRONG: only null is valid; "alex@co.com" would fail type validation
  type: 'null',
};

const correctSyntax = {
  // CORRECT: string or null are both valid
  type: ['string', 'null'],
};

console.log('Type syntax reference:');
console.log('  "type": "null"              → only null valid (almost never what you want)');
console.log('  "type": ["string",  "null"] → string or null  ✓');
console.log('  "type": ["number",  "null"] → number or null  ✓');
console.log('  "type": ["integer", "null"] → integer or null ✓');
console.log('  "type": ["boolean", "null"] → boolean or null ✓');
console.log('');
console.log('Exam rule: always use the union array form for nullable fields.');

## Stage 3: Format Normalization

Even with null handling in place, the same field can arrive in many natural language formats:

| Field | Natural language variants | Downstream needs |
|-------|--------------------------|------------------|
| `expected_salary` | `"fifty thousand"`, `"$5k/month"`, `"80K"` | `number` (annual, USD) |
| `years_experience` | `"several years"`, `"5+ years"`, `"half a decade"` | `integer` |
| `start_date` | `"next Monday"`, `"July 15th"`, `"ASAP"` | `string` (YYYY-MM-DD) or `null` |

**`description` is the only place Claude reads format rules.**  
The `type` field only declares what types are legal — it says nothing about
how to convert `"fifty thousand"` into `50000`. That conversion instruction
must live in `description`.

> **Schema vs Prompt division**  
> `input_schema` (type, enum, required) → controls **structure**  
> `description` + system prompt → controls **semantics** (format rules, null declaration)

In [ ]:
// Stage 3: format rules embedded in description fields
const normalizedApplicantTool: Anthropic.Tool = {
  name: 'extract_applicant',
  description: 'Extracts applicant information from job application text.',
  input_schema: {
    type: 'object',
    properties: {
      name: {
        type: ['string', 'null'],
        description: 'Applicant full name',
      },
      email: {
        type: ['string', 'null'],
        description: 'Contact email address',
      },
      years_experience: {
        type: ['integer', 'null'],
        description:
          "Years of professional experience as an integer. " +
          "If vague (e.g. 'several years'), use the midpoint (e.g. 3). " +
          "If not mentioned, return null.",
      },
      expected_salary: {
        type: ['number', 'null'],
        description:
          "Expected annual salary in USD as a number. " +
          "Convert natural language: 'fifty thousand' -> 50000, '$5k/month' -> 60000, '80K' -> 80000. " +
          "If not mentioned, return null.",
      },
      start_date: {
        type: ['string', 'null'],
        description:
          "Earliest available start date in YYYY-MM-DD format. " +
          "Resolve relative dates (e.g. 'next Monday') to absolute calendar dates. " +
          "Return null for 'ASAP' or if not mentioned.",
      },
    },
  },
};

// Natural language input — vague experience, informal salary, date in prose format
const vagueText =
  "Hi, I'm Jordan. I have several years in backend dev. " +
  "I'm looking for around 80k and can start July 15th.";

const stage3Response = await client.messages.create({
  model: MODEL,
  max_tokens: 512,
  system: SYSTEM_PROMPT_NULL,
  tools: [normalizedApplicantTool],
  tool_choice: { type: 'tool', name: 'extract_applicant' },
  messages: [{ role: 'user', content: vagueText }],
});

const stage3Block = stage3Response.content.find(b => b.type === 'tool_use');
if (stage3Block?.type === 'tool_use') {
  console.log('Stage 3 output (natural language normalised to machine-readable values):');
  console.log(JSON.stringify(stage3Block.input, null, 2));
  console.log('\n("several years" -> 3, "80k" -> 80000, "July 15th" -> YYYY-MM-DD)');
}

## Stage 3 Extension: Enum + Nullable Fields

When a **categorical** field might be absent, `enum` and nullable type must both include `null`:

```json
// WRONG — cannot return null even if field is absent
{
  "priority": {
    "type": "string",
    "enum": ["high", "medium", "low"]
  }
}

// CORRECT — null is listed in both type (union) and enum (value list)
{
  "priority": {
    "type": ["string", "null"],
    "enum": ["high", "medium", "low", null]
  }
}
```

**Two-layer rule**: `null` must appear in **both** places:

| Layer | Where | What it does |
|-------|-------|-------------|
| `"type": ["string", "null"]` | `type` array | Makes `null` a legal *type* |
| `null` in `"enum": [...]` | `enum` array | Makes `null` a legal *value* |

Missing either layer breaks the schema:
- `"type": "string"` + `null` in enum → type mismatch (`null` is not a `string`)
- `"type": ["string", "null"]` + no `null` in enum → enum constraint violation (`null` not in allowed list)

In [ ]:
// Enum + nullable: null must appear in BOTH type (union array) AND enum (value list)

const ticketTool: Anthropic.Tool = {
  name: 'extract_ticket',
  description: 'Extract support ticket information.',
  input_schema: {
    type: 'object',
    properties: {
      ticket_id: {
        type: 'string',
        description: 'Ticket identifier',
      },
      priority: {
        type: ['string', 'null'],
        enum: ['high', 'medium', 'low', null],   // null in both type AND enum
        description: 'Ticket priority level, or null if not specified in the request',
      },
      category: {
        type: ['string', 'null'],
        enum: ['billing', 'technical', 'account', 'other', null],
        description: 'Issue category, or null if unclear from the description',
      },
    },
    required: ['ticket_id'],
  },
};

const ticketText = 'Ticket T-1042: user cannot log in to their account. Submitted 2026-06-12.';

const ticketResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 256,
  system: SYSTEM_PROMPT_NULL,
  tools: [ticketTool],
  tool_choice: { type: 'tool', name: 'extract_ticket' },
  messages: [{ role: 'user', content: ticketText }],
});

const ticketBlock = ticketResponse.content.find(b => b.type === 'tool_use');
if (ticketBlock?.type === 'tool_use') {
  console.log('Enum + nullable output:');
  console.log(JSON.stringify(ticketBlock.input, null, 2));
  console.log('\n(priority: null — not mentioned; category: "technical" or "account" — login issue)');
}

## Comparing All Three Stages

Run the same ambiguous input through all three schemas side-by-side.

In [ ]:
const ambiguousText =
  "My name is Sam Rivera. I have several years of Python experience. Looking for 90K.";

async function runStage(
  label: string,
  tool: Anthropic.Tool,
  systemPrompt?: string
): Promise<void> {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 512,
    ...(systemPrompt ? { system: systemPrompt } : {}),
    tools: [tool],
    tool_choice: { type: 'tool', name: 'extract_applicant' },
    messages: [{ role: 'user', content: ambiguousText }],
  });
  const block = response.content.find(b => b.type === 'tool_use');
  if (block?.type === 'tool_use') {
    console.log(`\n=== ${label} ===`);
    console.log(JSON.stringify(block.input, null, 2));
  }
}

await runStage('Stage 1 — Base Prompt (fabrication risk)',     baseApplicantTool);
await runStage('Stage 2 — Null Handling (honest absence)',     nullableApplicantTool, SYSTEM_PROMPT_NULL);
await runStage('Stage 3 — Format Normalization (parsed)',      normalizedApplicantTool, SYSTEM_PROMPT_NULL);

## Summary

### Three-stage progression

| Stage | Change made | Problem solved |
|-------|------------|----------------|
| 1 Base Prompt | `type: 'string'` + `required` on all fields | Basic extraction |
| 2 Null Handling | `type: ['string', 'null']` + system rule | Fabrication on missing fields |
| 3 Format Normalization | Conversion instructions in `description` | Inconsistent natural language formats |

### Schema vs Prompt: who does what

```
JSON Schema (type, enum, required)   →  structure: what types are legal, which fields are mandatory
description + system prompt          →  semantics: how to fill, format rules, null declaration
```

These two layers cannot substitute for each other:
- `type: ['string', 'null']` makes null *legal* — it does not tell Claude *when* to use it
- A system prompt saying "return null if missing" does not make the type safe without the schema allowing null

### `required` guidance

| Scenario | Recommendation |
|----------|---------------|
| Field is a database primary key — must always exist | Add to `required` |
| Field may legitimately be absent in source documents | Omit from `required`, use `type: ['X', 'null']` |

### `null` vs Placeholder Strings

| Approach | Passes type check | Detectable downstream | Risk |
|----------|------------------|-----------------------|------|
| `null` | ✅ (with union type) | `=== null` always works | None |
| `"UNKNOWN"` / `""` | ✅ passes as `string` | Must inspect content | Silent contamination |
| Fabricated value | ✅ | Cannot detect without domain knowledge | Enters production undetected |

### Nullable Type Syntax

| Syntax | Valid outputs | Use case |
|--------|--------------|----------|
| `"type": "null"` | Only `null` | Rarely needed |
| `"type": ["string", "null"]` | String or `null` ✓ | Nullable text field |
| `"type": ["number", "null"]` | Number or `null` ✓ | Nullable numeric field |
| `"type": ["integer", "null"]` | Integer or `null` ✓ | Nullable count field |

### Enum + Nullable (two-layer rule)

`null` must appear in **both** `type` (union array) and `enum` (value list):

```json
{
  "priority": {
    "type": ["string", "null"],
    "enum": ["high", "medium", "low", null]
  }
}
```

Missing either layer: `"type": "string"` + `null` in enum → type mismatch;
`"type": ["string", "null"]` + no `null` in enum → enum violation.

### Related patterns

- **Schema Redundancy** (`calculated_total` vs `stated_total`) — detects inconsistencies *between*
  extracted values (arithmetic or logical), complementary to null handling
- **Resilient Catch-all** (`extra_metadata` container) — handles unexpected *extra* fields
  when document structure is unknown or expanding